# Operational Schema - Lakebase Plant Domain
This notebook demonstrates the operational schema in the `plant` schema of the ironbark Lakebase instance.
It shows:
- Connection to Lakebase
- Tables and their columns
- Primary, foreign, and check constraints
- Indexes
- A multi-table join proving the relationships work
- Row counts for validation

In [1]:
import sys
sys.path.insert(0, '/Users/chris.dorrington/llm/ai-day')
from lakebase.lb import connect

conn, host = connect(branch="production")
print(f"Connected to host: {host}")

# Show version
cur = conn.cursor()
cur.execute("SELECT version()")
version = cur.fetchone()[0]
print(f"PostgreSQL Version: {version}")


Connected to host: ep-empty-poetry-d2h5n6t2.database.us-east-1.cloud.databricks.com


PostgreSQL Version: PostgreSQL 17.11 (df1f1a3) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit


In [2]:
# List all tables in the plant schema with column count
sql = '''
    SELECT
        t.tablename,
        COUNT(a.attname) as column_count
    FROM pg_tables t
    LEFT JOIN pg_class c ON c.relname = t.tablename
    LEFT JOIN pg_attribute a ON a.attrelid = c.oid AND a.attnum > 0
    WHERE t.schemaname = 'plant'
    GROUP BY t.tablename
    ORDER BY t.tablename
'''
cur.execute(sql)

tables = cur.fetchall()
print(f"Tables in 'plant' schema: {len(tables)}")
print()
for table_name, col_count in tables:
    print(f"  {table_name:<30} - {col_count} columns")


Tables in 'plant' schema: 4

  alert_outbox                   - 21 columns
  tag_current                    - 11 columns
  work_order                     - 17 columns
  work_order_note                - 8 columns


In [3]:
# PRIMARY KEY constraints
sql = '''
    SELECT
        t.relname as table_name,
        pg_get_constraintdef(c.oid) as constraint_definition
    FROM pg_constraint c
    JOIN pg_class t ON t.oid = c.conrelid
    JOIN pg_namespace n ON n.oid = t.relnamespace
    WHERE n.nspname = 'plant'
        AND c.contype = 'p'
    ORDER BY t.relname
'''
cur.execute(sql)

pk_constraints = cur.fetchall()
print(f"PRIMARY KEY constraints ({len(pk_constraints)}):")
print()
for table_name, constraint_def in pk_constraints:
    print(f"  {table_name}: {constraint_def}")


PRIMARY KEY constraints (4):

  alert_outbox: PRIMARY KEY (alert_id)
  tag_current: PRIMARY KEY (tag_id)
  work_order: PRIMARY KEY (work_order_id)
  work_order_note: PRIMARY KEY (note_id)


In [4]:
# FOREIGN KEY constraints
sql = '''
    SELECT
        t.relname as table_name,
        pg_get_constraintdef(c.oid) as constraint_definition
    FROM pg_constraint c
    JOIN pg_class t ON t.oid = c.conrelid
    JOIN pg_namespace n ON n.oid = t.relnamespace
    WHERE n.nspname = 'plant'
        AND c.contype = 'f'
    ORDER BY t.relname
'''
cur.execute(sql)

fk_constraints = cur.fetchall()
print(f"FOREIGN KEY constraints ({len(fk_constraints)}):")
print()
for table_name, constraint_def in fk_constraints:
    print(f"  {table_name}: {constraint_def}")


FOREIGN KEY constraints (2):

  work_order: FOREIGN KEY (alert_id) REFERENCES plant.alert_outbox(alert_id)
  work_order_note: FOREIGN KEY (work_order_id) REFERENCES plant.work_order(work_order_id) ON DELETE CASCADE


In [5]:
# CHECK constraints
sql = '''
    SELECT
        t.relname as table_name,
        pg_get_constraintdef(c.oid) as constraint_definition
    FROM pg_constraint c
    JOIN pg_class t ON t.oid = c.conrelid
    JOIN pg_namespace n ON n.oid = t.relnamespace
    WHERE n.nspname = 'plant'
        AND c.contype = 'c'
    ORDER BY t.relname
'''
cur.execute(sql)

ck_constraints = cur.fetchall()
print(f"CHECK constraints ({len(ck_constraints)}):")
print()
if ck_constraints:
    for table_name, constraint_def in ck_constraints:
        print(f"  {table_name}: {constraint_def}")
else:
    print("  (none)")


CHECK constraints (4):

  work_order: CHECK (((status = 'CLOSED'::text) = (closed_at IS NOT NULL)))
  work_order: CHECK ((priority = ANY (ARRAY['CRITICAL'::text, 'HIGH'::text, 'MEDIUM'::text, 'LOW'::text])))
  work_order: CHECK ((status = ANY (ARRAY['OPEN'::text, 'IN_PROGRESS'::text, 'ON_HOLD'::text, 'CLOSED'::text, 'CANCELLED'::text])))
  work_order_note: CHECK ((note_kind = ANY (ARRAY['DIAGNOSIS'::text, 'ACTION'::text, 'PARTS'::text, 'HANDOVER'::text, 'SAFETY'::text])))


In [6]:
# Indexes on plant schema
sql = '''
    SELECT
        t.relname as table_name,
        i.relname as index_name,
        ix.indisprimary,
        ix.indisunique,
        pg_get_indexdef(ix.indexrelid) as index_definition
    FROM pg_index ix
    JOIN pg_class t ON t.oid = ix.indrelid
    JOIN pg_class i ON i.oid = ix.indexrelid
    JOIN pg_namespace n ON n.oid = t.relnamespace
    WHERE n.nspname = 'plant'
    ORDER BY t.relname, i.relname
'''
cur.execute(sql)

indexes = cur.fetchall()
print(f"Indexes on plant schema ({len(indexes)}):")
print()
for table_name, index_name, is_pk, is_unique, index_def in indexes:
    type_str = "PRIMARY" if is_pk else ("UNIQUE" if is_unique else "INDEX")
    print(f"  {table_name}.{index_name} ({type_str})")
    print(f"    {index_def}")


Indexes on plant schema (17):

  alert_outbox.alert_outbox_pkey (PRIMARY)
    CREATE UNIQUE INDEX alert_outbox_pkey ON plant.alert_outbox USING btree (alert_id)
  alert_outbox.ix_alert_open (INDEX)
    CREATE INDEX ix_alert_open ON plant.alert_outbox USING btree (raised_at DESC) WHERE (acknowledged_at IS NULL)
  tag_current.ix_tag_current_quality (INDEX)
    CREATE INDEX ix_tag_current_quality ON plant.tag_current USING btree (quality) WHERE (quality <> 'GOOD'::text)
  tag_current.ix_tag_current_source_ts (INDEX)
    CREATE INDEX ix_tag_current_source_ts ON plant.tag_current USING btree (source_ts DESC)
  tag_current.tag_current_pkey (PRIMARY)
    CREATE UNIQUE INDEX tag_current_pkey ON plant.tag_current USING btree (tag_id)
  work_order.ix_fts_work_order_description (INDEX)
    CREATE INDEX ix_fts_work_order_description ON plant.work_order USING gin (to_tsvector('english'::regconfig, COALESCE(description, ''::text)))
  work_order.ix_fts_work_order_resolution_notes (INDEX)
    CREATE I

In [7]:
# 3-table JOIN: alert_outbox -> work_order -> work_order_note
# Showing: severity, work_order, asset, failure_mode, note_count

sql = '''
    SELECT
        a.severity,
        wo.work_order_id,
        wo.asset_id,
        wo.failure_mode,
        COUNT(won.note_id) as note_count
    FROM plant.alert_outbox a
    JOIN plant.work_order wo ON wo.alert_id = a.alert_id
    LEFT JOIN plant.work_order_note won ON won.work_order_id = wo.work_order_id
    GROUP BY a.severity, wo.work_order_id, wo.asset_id, wo.failure_mode
    ORDER BY a.severity, wo.work_order_id
    LIMIT 20
'''
cur.execute(sql)

results = cur.fetchall()
print(f"3-table JOIN results ({len(results)} rows shown):")
print()
print("Severity | Work Order | Asset ID | Failure Mode           | Note Count")
print("-" * 80)
for severity, wo_id, asset_id, failure_mode, note_count in results:
    print(f"{str(severity):<8} | {str(wo_id):<10} | {str(asset_id):<8} | {str(failure_mode):<22} | {note_count}")


3-table JOIN results (18 rows shown):

Severity | Work Order | Asset ID | Failure Mode           | Note Count
--------------------------------------------------------------------------------
CRITICAL | WO-1001    | CRU-PCR01 | pressure               | 2
CRITICAL | WO-1006    | CV-CV002 | motor_current          | 2
CRITICAL | WO-1009    | ROM-APF01 | belt_speed             | 3
CRITICAL | WO-1013    | CV-CV007 | motor_current          | 2
CRITICAL | WO-1016    | ROM-BIN01 | level                  | 2
HIGH     | WO-1015    | ROM-BIN01 | level                  | 2
WARNING  | WO-1002    | STK-STK01 | mass_flow              | 2
WARNING  | WO-1003    | CV-CV006 | mass_flow              | 2
WARNING  | WO-1004    | CV-CV001 | mass_flow              | 2
WARNING  | WO-1005    | DSD-CYC01 | volume_flow            | 2
WARNING  | WO-1007    | CV-CV008 | motor_current          | 2
WARNING  | WO-1008    | ROM-APF01 | belt_speed             | 2
WARNING  | WO-1010    | LAB-ROM01 | moisture              

In [8]:
# Row counts for each table in plant schema
cur.execute("SELECT COUNT(*) FROM plant.work_order")
wo_count = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM plant.alert_outbox")
ao_count = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM plant.work_order_note")
won_count = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM plant.tag_current")
tc_count = cur.fetchone()[0]

print(f"Row counts in plant schema:")
print()
print("Table Name                     | Row Count")
print("-" * 50)
print(f"{'alert_outbox':<30} | {ao_count}")
print(f"{'tag_current':<30} | {tc_count}")
print(f"{'work_order':<30} | {wo_count}")
print(f"{'work_order_note':<30} | {won_count}")
print("-" * 50)
print(f"{'TOTAL':<30} | {ao_count + tc_count + wo_count + won_count}")

# Store for later
_plant_counts_start = {
    'work_order': wo_count,
    'alert_outbox': ao_count,
    'work_order_note': won_count,
    'tag_current': tc_count
}


Row counts in plant schema:

Table Name                     | Row Count
--------------------------------------------------
alert_outbox                   | 25
tag_current                    | 0
work_order                     | 22
work_order_note                | 49
--------------------------------------------------
TOTAL                          | 96
